# Paper compare — entrenamiento GPU en Colab

Pipeline completo de `paper_compare_latent_clusters`: AE + encoder supervisado + clustering + BFPC / Inception / ViT.

**Antes de Run all**
1. Runtime → Cambiar tipo de entorno → **GPU**
2. En Drive: `MyDrive/retina_paper_compare/data/rfmid/` con RFMiD
3. ZIP de código en `MyDrive/retina_paper_compare/code/paper_compare_colab_bundle.zip` (generado con `python scripts/make_colab_bundle.py`)

Sin secretos: solo rutas de Drive. Guía: `colab/README_COLAB.md`.


## 0. Montar Drive + comprobar GPU


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, time, zipfile

import torch
from google.colab import drive

drive.mount("/content/drive")

print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU no disponible. Ve a Entorno de ejecución → Cambiar tipo de entorno → GPU y vuelve a ejecutar."
    )
print("GPU:", torch.cuda.get_device_name(0))


## 1. Configuración de rutas (edita si tu Drive difiere)


In [ ]:
# === Edita solo estas rutas si hace falta (sin tokens / secretos) ===
DRIVE_ROOT = Path("/content/drive/MyDrive/retina_paper_compare")
DRIVE_CODE_ZIP = DRIVE_ROOT / "code" / "paper_compare_colab_bundle.zip"
DRIVE_DATA = DRIVE_ROOT / "data" / "rfmid"          # Training_Set / Evaluation_Set / Test_Set
DRIVE_OUTPUTS = DRIVE_ROOT / "outputs"

PROJECT_ROOT = Path("/content/retina_multilabel")
EXP = PROJECT_ROOT / "experiments" / "paper_compare_latent_clusters"

# Horarios paper-ish (100 es el paper; 30 es un buen compromiso T4)
AE_EPOCHS = 20
SUP_EPOCHS = 10
PAPER_EPOCHS = 30          # BFPC / Inception / ViT — sube a 100 si tienes cuota
OPT_EPOCHS = 15            # Stage 5: LR sweep del winner (GPU)
BATCH_SIZE_AE = 32
BATCH_SIZE_SUP = 16
BATCH_SIZE_PAPER = 16
NUM_WORKERS = 2
SKIP_DONE = True           # reanudar tras desconexión Colab
TRAIN_LATENT_MAPS = True   # además de ODIR-8, entrena en el mejor mapa B/C
RUN_COMPARE_OPTIMIZE = True  # Stage 5: 06 + 08 tras trains paper
LATENT_TAXONOMY = "B"      # "B" (autoencoder) o "C" (supervised)

for d in (DRIVE_ROOT / "code", DRIVE_DATA, DRIVE_OUTPUTS):
    d.mkdir(parents=True, exist_ok=True)

assert DRIVE_DATA.is_dir(), f"Sube RFMiD a {DRIVE_DATA}"
assert any(DRIVE_DATA.rglob("RFMiD_Training_Labels.csv")), (
    f"No se encontró RFMiD_Training_Labels.csv bajo {DRIVE_DATA}"
)
print("Drive OK:", DRIVE_ROOT)
print("Data sample:", next(DRIVE_DATA.rglob("RFMiD_Training_Labels.csv")))


## 2. Instalar código desde el ZIP de Drive


In [ ]:
assert DRIVE_CODE_ZIP.is_file(), (
    f"Falta {DRIVE_CODE_ZIP}. En local: python scripts/make_colab_bundle.py "
    "y súbelo a Drive/code/."
)

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True)

with zipfile.ZipFile(DRIVE_CODE_ZIP) as zf:
    zf.extractall(PROJECT_ROOT)

# Si el zip empaquetó con prefijo extra, busca pyproject.toml
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    matches = list(PROJECT_ROOT.rglob("pyproject.toml"))
    if not matches:
        raise FileNotFoundError("pyproject.toml no está en el ZIP")
    nested = matches[0].parent
    if nested != PROJECT_ROOT:
        for child in nested.iterdir():
            dest = PROJECT_ROOT / child.name
            if dest.exists():
                continue
            shutil.move(str(child), str(dest))

assert (PROJECT_ROOT / "pyproject.toml").is_file()
assert EXP.is_dir(), EXP

%pip install -q --no-deps -e /content/retina_multilabel
%pip install -q "timm>=1.0" scikit-learn pandas matplotlib pillow tqdm huggingface_hub

os.environ["RETINA_DATA_ROOT"] = str(DRIVE_DATA)
os.environ["TORCH_HOME"] = str(DRIVE_ROOT / "torch_cache")
Path(os.environ["TORCH_HOME"]).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(EXP))

print("PROJECT_ROOT", PROJECT_ROOT)
print("EXP", EXP)
print("RETINA_DATA_ROOT", os.environ["RETINA_DATA_ROOT"])


## Helpers: ejecutar scripts y sincronizar a Drive


In [ ]:
def run_script(rel_script: str, *args: str, cwd=None) -> None:
    script = EXP / "scripts" / rel_script
    cmd = [sys.executable, str(script), *args]
    print("\n>>>", " ".join(cmd), flush=True)
    t0 = time.time()
    subprocess.run(cmd, check=True, cwd=str(cwd or PROJECT_ROOT))
    print(f"<<< done in {time.time() - t0:.1f}s", flush=True)


def sync_outputs(label: str = "") -> None:
    # Copia EXP/outputs -> Drive (merge; no borra mapas ajenos en Drive).
    src = EXP / "outputs"
    src.mkdir(parents=True, exist_ok=True)
    dest = DRIVE_OUTPUTS
    dest.mkdir(parents=True, exist_ok=True)
    for path in src.rglob("*"):
        if path.is_dir():
            continue
        rel = path.relative_to(src)
        out = dest / rel
        out.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, out)
    print(f"[sync] {label} -> {dest}")


def already_done(*markers) -> bool:
    return SKIP_DONE and all(m.is_file() for m in markers)


# Restaurar outputs previos desde Drive (reanudacion)
if DRIVE_OUTPUTS.exists() and any(DRIVE_OUTPUTS.iterdir()):
    for path in DRIVE_OUTPUTS.rglob("*"):
        if path.is_dir():
            continue
        rel = path.relative_to(DRIVE_OUTPUTS)
        out = EXP / "outputs" / rel
        if out.exists() and SKIP_DONE:
            continue
        out.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, out)
    print("Restaurados outputs desde Drive (si habia).")


## 3a. Taxonomía ODIR-8 + clustering A (ResNet ImageNet)


In [ ]:
odir_map = EXP / "outputs" / "odir8_taxonomy" / "cluster_map.json"
a_map = EXP / "outputs" / "clustering_A_baseline" / "cluster_map.json"

if not already_done(odir_map):
    run_script("01b_build_odir_taxonomy.py")
else:
    print("SKIP ODIR-8 (ya existe)")

if not already_done(a_map):
    run_script("01_run_baseline_clustering.py", "--device", "cuda", "--batch-size", "32")
else:
    print("SKIP clustering A (ya existe)")

sync_outputs("odir8+A")


## 3b. AE completo (`01c`) + cluster (`01d`)


In [ ]:
ae_ckpt = EXP / "outputs" / "clustering_B_autoencoder" / "checkpoints" / "autoencoder.pt"
b_map = EXP / "outputs" / "clustering_B_autoencoder" / "cluster_map.json"

need_ae = not ae_ckpt.is_file()
if ae_ckpt.is_file() and SKIP_DONE:
    import json as _json
    summary = EXP / "outputs" / "clustering_B_autoencoder" / "checkpoints" / "train_summary.json"
    if summary.is_file():
        meta = _json.loads(summary.read_text())
        if int(meta.get("epochs", 0)) < AE_EPOCHS or int(meta.get("n_images", 0)) < 1000:
            need_ae = True
            print("AE previo es smoke/CPU → reentrenar completo en GPU")
        else:
            need_ae = False

if need_ae:
    run_script(
        "01c_train_autoencoder.py",
        "--epochs", str(AE_EPOCHS),
        "--batch-size", str(BATCH_SIZE_AE),
        "--device", "cuda",
    )
else:
    print("SKIP AE train")

if need_ae or not b_map.is_file():
    run_script("01d_cluster_autoencoder.py", "--device", "cuda")
else:
    print("SKIP AE cluster")

sync_outputs("AE+B")


## 3c. Encoder supervisado (`01e`) + cluster (`01f`)


In [ ]:
sup_ckpt = EXP / "outputs" / "clustering_C_supervised" / "checkpoints" / "supervised_encoder.pt"
c_map = EXP / "outputs" / "clustering_C_supervised" / "cluster_map.json"

need_sup = not sup_ckpt.is_file()
if sup_ckpt.is_file() and SKIP_DONE:
    import json as _json
    summary = EXP / "outputs" / "clustering_C_supervised" / "checkpoints" / "train_summary.json"
    if summary.is_file():
        meta = _json.loads(summary.read_text())
        if int(meta.get("epochs", 0)) < SUP_EPOCHS or int(meta.get("n_images", 0)) < 1000:
            need_sup = True
            print("Supervised previo es smoke/CPU → reentrenar completo en GPU")
        else:
            need_sup = False

if need_sup:
    run_script(
        "01e_train_supervised_encoder.py",
        "--epochs", str(SUP_EPOCHS),
        "--batch-size", str(BATCH_SIZE_SUP),
        "--device", "cuda",
    )
else:
    print("SKIP supervised train")

if need_sup or not c_map.is_file():
    run_script("01f_cluster_supervised.py", "--device", "cuda")
else:
    print("SKIP supervised cluster")

sync_outputs("sup+C")


## 3d. Comparar clusterings A/B/C vs ODIR-8


In [ ]:
compare_md = EXP / "outputs" / "clustering_compare" / "COMPARE.md"
if not already_done(compare_md):
    run_script("compare_clusterings.py")
else:
    print("SKIP compare (ya existe)")
sync_outputs("compare")
print((EXP / "outputs" / "clustering_compare" / "COMPARE.md").read_text()[:1500])


## 4. Modelos paper: BFPC / Inception / ViT

Por defecto entrenan sobre **ODIR-8**. Si `TRAIN_LATENT_MAPS=True`, también sobre el mapa B o C.
Épocas: `PAPER_EPOCHS` (recomendado 30 en T4; 100 = protocolo paper).
ViT/DINOv2 usa **392²** (BaselineV2); BFPC 256 / Inception 299 siguen su paper.
Si OOM en T4 con ViT, baja `BATCH_SIZE_VIT` a 4.


In [ ]:
BATCH_SIZE_VIT = globals().get("BATCH_SIZE_VIT", 8)  # 392² needs less VRAM than BFPC/Inception

def train_paper_models(taxonomy_tag: str, cluster_map: Path, labels_dir: Path) -> None:
    common = [
        "--epochs", str(PAPER_EPOCHS),
        "--num-workers", str(NUM_WORKERS),
        "--patience", "8",
        "--cluster-map", str(cluster_map),
        "--labels-dir", str(labels_dir),
    ]
    jobs = [
        ("03_train_bfpc.py", f"bfpc_{taxonomy_tag}", ["--mode", "single", "--batch-size", str(BATCH_SIZE_PAPER)]),
        ("04_train_inception.py", f"inception_{taxonomy_tag}", ["--batch-size", str(BATCH_SIZE_PAPER)]),
        (
            "05_train_vit.py",
            f"vit_dinov2_{taxonomy_tag}",
            ["--backbone", "dinov2", "--img-size", "392", "--batch-size", str(BATCH_SIZE_VIT)],
        ),
    ]
    for script, run_name, extra in jobs:
        marker = EXP / "outputs" / "runs" / run_name / "run_summary.json"
        if already_done(marker):
            print(f"SKIP {run_name}")
            continue
        run_script(
            script,
            *common,
            "--run-name", run_name,
            *extra,
        )
        sync_outputs(run_name)


# ODIR-8 (primario paper)
odir_map = EXP / "outputs" / "odir8_taxonomy" / "cluster_map.json"
odir_labels = EXP / "outputs" / "odir8_taxonomy" / "labels"
assert odir_map.is_file() and (odir_labels / "clustered_train.csv").is_file()
train_paper_models("odir8", odir_map, odir_labels)

if TRAIN_LATENT_MAPS:
    if LATENT_TAXONOMY.upper() == "C":
        lat_map = EXP / "outputs" / "clustering_C_supervised" / "cluster_map.json"
        lat_labels = EXP / "outputs" / "clustering_C_supervised" / "labels"
        tag = "latentC"
    else:
        lat_map = EXP / "outputs" / "clustering_B_autoencoder" / "cluster_map.json"
        lat_labels = EXP / "outputs" / "clustering_B_autoencoder" / "labels"
        tag = "latentB"
    assert lat_map.is_file(), lat_map
    train_paper_models(tag, lat_map, lat_labels)

sync_outputs("paper_final")
print("Paper trains done. Resultados en", DRIVE_OUTPUTS)
print("Siguiente: celda Stage 5 (compare + optimize)")


## 5. Comparar modelos + optimizar winner

Agrega métricas TEST (`06`) y hace un sweep de LR del ganador (`08`, `OPT_EPOCHS`).


In [ ]:
OPT_EPOCHS = globals().get("OPT_EPOCHS", 15)
RUN_COMPARE_OPTIMIZE = globals().get("RUN_COMPARE_OPTIMIZE", True)
BATCH_SIZE_PAPER = globals().get("BATCH_SIZE_PAPER", 16)
sweep_marker = EXP / "outputs" / "comparison" / "optimization_sweep.json"

if not RUN_COMPARE_OPTIMIZE:
    print("SKIP compare/optimize (RUN_COMPARE_OPTIMIZE=False)")
elif already_done(sweep_marker):
    print("SKIP optimize (ya existe optimization_sweep.json)")
else:
    run_script("06_compare_and_optimize.py")
    hooks = EXP / "outputs" / "comparison" / "optimization_hooks.json"
    assert hooks.is_file(), hooks
    import json as _json
    h = _json.loads(hooks.read_text())
    print("Winner:", h.get("winner_run"), h.get("winner_model"), "AP=", h.get("winner_ap_macro"))
    if h.get("winner_is_smoke"):
        print("WARNING: winner is smoke — skipping 08 until paper runs exist")
    else:
        run_script(
            "08_optimize_winner.py",
            "--epochs", str(OPT_EPOCHS),
            "--batch-size", str(BATCH_SIZE_PAPER),
        )
        run_script("06_compare_and_optimize.py")

sync_outputs("compare_optimize")
print("comparison/:", list((EXP / "outputs" / "comparison").glob("*")))


## 6. Resumen rápido

Descarga `MyDrive/retina_paper_compare/outputs/` → `07_import_colab_outputs.py` + `09_verify_goal.py`.


In [ ]:
import json
from IPython.display import display, Markdown

lines = ["### Runs completados", ""]
runs_dir = EXP / "outputs" / "runs"
if runs_dir.is_dir():
    for summary in sorted(runs_dir.glob("*/run_summary.json")):
        data = json.loads(summary.read_text())
        tm = data.get("test_metrics", {})
        lines.append(
            f"- **{summary.parent.name}**: AP={tm.get('ap_macro')} AUROC={tm.get('auroc_macro')} "
            f"F1={tm.get('f1_macro')} (ep best={data.get('best_epoch')})"
        )
else:
    lines.append("_Sin runs aún._")

cmp_table = EXP / "outputs" / "comparison" / "COMPARISON.md"
if cmp_table.is_file():
    lines.append("")
    lines.append("### Model comparison")
    lines.append("")
    lines.append(cmp_table.read_text()[:2500])

sweep = EXP / "outputs" / "comparison" / "optimization_sweep.json"
if sweep.is_file():
    lines.append("")
    lines.append("### Optimization sweep")
    lines.append("```json")
    lines.append(sweep.read_text()[:1500])
    lines.append("```")

display(Markdown("\n".join(lines)))
print("Drive outputs:", DRIVE_OUTPUTS)
